# NetSentry — Network Anomaly Detector (Colab, no external windows)

**Just run the cells top to bottom.** Cell 1 will ask you to upload `network-anomaly-detector.zip`. Everything else — install, generate data, train the ML models, run detection, and the live dashboard — happens automatically and the dashboard renders **inline in this notebook**, no popup or new tab needed.

In [ ]:
# CELL 1 — Upload the zip
from google.colab import files
print("Select network-anomaly-detector.zip ...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f"Uploaded: {zip_name}")

In [ ]:
# CELL 2 — Extract + install dependencies
import zipfile, os

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# find the extracted project folder automatically
candidates = [d for d in os.listdir('.') if os.path.isdir(d) and 'network-anomaly' in d.lower()]
PROJECT_DIR = candidates[0] if candidates else 'network-anomaly-detector'
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

!pip install -q -r requirements.txt
print("Dependencies installed.")

In [ ]:
# CELL 3 — Generate synthetic traffic + train the models
!python3 src/data_generator.py
print()
!python3 src/train_model.py

In [ ]:
# CELL 4 — Run batch detection once, so the dashboard has data immediately
!python3 src/detector.py

In [ ]:
# CELL 5 — Start the live traffic simulator in the background
# (keeps feeding new simulated flows + alerts continuously)
import subprocess, sys, time

sim_proc = subprocess.Popen(
    [sys.executable, "src/live_simulator.py", "--delay", "0.3"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print(f"Live simulator running in background (pid {sim_proc.pid}). It will keep generating traffic + alerts.")

In [ ]:
# CELL 6 — Start the dashboard server in the background
import subprocess, sys, time

dash_proc = subprocess.Popen(
    [sys.executable, "dashboard/app.py"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**__import__('os').environ, "FLASK_DEBUG": "0"}
)
time.sleep(3)
print(f"Dashboard server running in background (pid {dash_proc.pid}).")

In [ ]:
# CELL 7 — Embed the live dashboard directly in this notebook (no new window)
from google.colab.output import eval_js
from IPython.display import IFrame, display

proxy_url = eval_js("google.colab.kernel.proxyPort(5000)")
print("Dashboard URL (also embedded below):", proxy_url)
display(IFrame(src=proxy_url, width=1100, height=750))

---
### Re-run the dashboard view
If the embedded view above goes blank after a while (Colab sometimes drops the proxy iframe), just re-run **Cell 7** — no need to restart the servers.

### Stop everything
Run the cell below when you're done, or just "Restart runtime".

In [ ]:
# CELL 8 — Stop background processes
try:
    sim_proc.terminate()
    dash_proc.terminate()
    print("Stopped simulator and dashboard.")
except NameError:
    print("Nothing running.")